# Day 20: Relational Recurrent Neural Networks (RMC)

## Interactive Walkthrough

In this notebook, we will:
1.  **Build the Relational Memory Core** of Santoro et al. (2018), Sec. 3, and take one step by hand.
2.  **Look at the Nth Farthest task** (Sec. 4.1, App. A.1) and at the RMC's $N \times (N+1)$ attention on one sequence.
3.  **Build an LSTM baseline** with the same number of parameters.
4.  **Train both briefly** and read the accuracy against the right baselines.

### Why this matters
An LSTM keeps everything in one vector. The RMC keeps a few memory **slots** and lets them attend to each other and to the new input at every step, so it can relate a new observation to what it already stored.

In [ ]:
import os, tempfile
import torch
from IPython.display import Image, display

from implementation import RelationalMemory, RelationalRNN, StandardLSTM, count_parameters
from visualization import plot_attention_heatmap, plot_comparison
from train_minimal import (generate_nth_farthest_batch, build_parser, build_models,
                           train_model, question_n)

torch.manual_seed(0)
FIG_DIR = tempfile.mkdtemp()   # figures go to a temporary folder, not into the repo

## 1. The Relational Memory Core

One step, for memory $M$ ($N$ slots of width $F$) and input $x_t$ (Fig. 1a):

1. $\tilde x = W_{in} x_t$: the input becomes **one extra row**.
2. Multi-head attention over $[M; \tilde x]$ (Eq. 2): queries from the $N$ slots, keys and values from all $N+1$ rows, so the attention matrix is $N \times (N+1)$ and the memory keeps its size.
3. Residual + LayerNorm, then a row-wise MLP with residual + LayerNorm: the result is $G$.
4. Gates per row, $[i; f] = \sigma(W \tilde x + U \tanh(M_{t-1}) + [0; 1])$, and $M_t = i \odot \tanh(G) + f \odot M_{t-1}$. There is no output gate.

In [ ]:
INPUT_SIZE = 11     # D + 3K for the task below: 2-dim vectors, 3 labels, n and m one-hot
MEM_SLOTS = 4
MEM_SIZE = 64
HEADS = 4

rmc = RelationalMemory(mem_slots=MEM_SLOTS, mem_size=MEM_SIZE, input_size=INPUT_SIZE, num_heads=HEADS)
print(f"RMC core: {MEM_SLOTS} slots of size {MEM_SIZE}, {count_parameters(rmc):,} parameters")

# The initial memory is the identity, zero-padded to N x F (Sonnet): every slot starts different
print("initial memory, first 6 columns:\n", rmc.initial_memory[0, :, :6])

x = torch.randn(1, INPUT_SIZE)
memory, attention = rmc(x)          # memory_state=None -> start from the initial memory
print("next memory:", tuple(memory.shape), "  attention:", tuple(attention.shape), "= (batch, heads, N, N+1)")
print("rows sum to 1:", torch.allclose(attention.sum(-1), torch.ones(1, HEADS, MEM_SLOTS)))
print("how much each slot reads the input (last column, mean over heads):",
      attention[0, :, :, -1].mean(0).detach().numpy().round(3))

Because the weights are shared across slots, more slots add capacity but no parameters:

In [ ]:
for n_slots in (1, 4, 16):
    core = RelationalMemory(mem_slots=n_slots, mem_size=MEM_SIZE, input_size=INPUT_SIZE, num_heads=HEADS)
    print(f"{n_slots:2d} slots: {count_parameters(core):,} parameters, {n_slots * MEM_SIZE} memory units")

## 2. The Nth Farthest task

Each sequence shows $K$ random vectors one at a time. Each vector carries a random label, and every step repeats the question: "which vector is the $n$-th farthest from vector $m$?" The answer is a label. The paper uses $K = 8$ vectors of 16 dims; we use a small version, 3 vectors of 2 dims, so it trains on a CPU in minutes.

Counting is 0-based: $n = 0$ asks for the farthest vector, and $n = K-1$ for the nearest, which is vector $m$ itself.

In [ ]:
K, D = 3, 2
inputs, targets = generate_nth_farthest_batch(1, K, D, torch.Generator().manual_seed(3))
seq = inputs[0]
labels = seq[:, D:D + K].argmax(1)
n, m = seq[0, D + K:D + 2 * K].argmax().item(), seq[0, D + 2 * K:].argmax().item()
print(f"question: which vector is rank n = {n} (0 = farthest) from vector m = {m}?")
ref = seq[labels.tolist().index(m), :D]
for t in range(K):
    print(f"  step {t}: label {labels[t].item()}  x = {seq[t, :D].numpy().round(2)}  "
          f"distance to vector {m} = {(seq[t, :D] - ref).norm().item():.3f}")
print("answer (target label):", targets[0].item())

## 3. Attention over a sequence

Unroll an (untrained) RMC over the sequence and plot each step's attention, averaged over heads. Rows are the slots asking; columns are the slots answering plus the input row ("in"). The step where vector $m$ arrives is marked. In the trained models of the paper (Fig. 3), slots read the input heavily before $m$ arrives and focus on the slots holding $m$ afterwards.

In [ ]:
rnn = RelationalRNN(INPUT_SIZE, MEM_SLOTS, MEM_SIZE, K, num_heads=HEADS)
with torch.no_grad():
    _, attn = rnn(inputs, return_attention=True)          # (batch, T, heads, N, N+1)
path = os.path.join(FIG_DIR, "attention.png")
plot_attention_heatmap(attn[0], path, highlight_step=labels.tolist().index(m),
                       title=f"Untrained RMC: rank n = {n} from vector {m}?")
display(Image(path))

## 4. An LSTM baseline with the same number of parameters

Both models feed their final state to the same output MLP (the paper uses 4 layers of 256 units for every model). `build_models` picks the LSTM hidden size that matches the RMC's total parameter count.

In [ ]:
args = build_parser().parse_args(['--log_every', '500'])   # train_minimal.py defaults
rmc_model, lstm_model = build_models(args)

## 5. A short training run

What to expect: when $n = K-1$ the answer is $m$, which is in the input. Copying $m$ for those questions and guessing among the other $K-1$ labels otherwise gives $2/K$ accuracy with no reasoning at all. Every other question needs the relational part, and there the uniform $n$ hides all signal: averaged over $n$, every label other than $m$ is equally likely whatever the vectors are, so a model has to combine $n$ with the distances before it can improve. Gradient descent finds that combination slowly; the paper and a public PyTorch reproduction report hours on a GPU at 25% for the 8-vector task.

The run below uses `train_minimal.py`'s defaults: 3 vectors of 2 dims and a short curriculum (the first 1500 of 3000 steps ask only for the farthest vector; the paper samples every $n$ from the start), with evaluation on every $n$. It takes about 2 minutes on one CPU core. In our runs (3 seeds) the RMC reached 94.5-95.4% and the parameter-matched LSTM 67.3-67.7%: the copy-$m$ level, with chance on the relational questions. With `--curriculum_steps 0` (the paper's protocol) both models stay at 67.7%.

In [ ]:
eval_set = generate_nth_farthest_batch(args.eval_size, args.num_objects, args.num_features, torch.Generator().manual_seed(12345))
results = {name: train_model(name, model, args, eval_set) for name, model in (('RMC', rmc_model), ('LSTM', lstm_model))}
K = args.num_objects
for name, r in results.items():
    print(f"{name}: accuracy {r['before']['acc']:.3f} -> {r['after']['acc']:.3f}; relational questions "
          f"{r['after']['acc_relational']:.3f} (chance {1 / (K - 1):.3f}); copy-m questions {r['after']['acc_reference']:.3f}")
print(f"baselines: chance {1 / K:.3f}, copying m when n = K-1: {2 / K:.3f}")
path = os.path.join(FIG_DIR, "curves.png")
plot_comparison({name: r['history'] for name, r in results.items()}, path,
                baselines={'acc_relational': {'chance 1/(K-1)': 1 / (K - 1)},
                           'acc': {'chance 1/K': 1 / K, 'copy m: 2/K': 2 / K}},
                marker_step=args.curriculum_steps)
display(Image(path))

## Takeaways

- The RMC's state is a set of slots; one attention over $[M; \tilde x]$ both mixes the slots and writes the input, and its extra column shows when each slot reads the input.
- Parameters are shared across slots, so capacity ($N \times F$) and parameter count are separate knobs.
- On Nth Farthest, read the accuracy against $2/K$ and look at the relational questions separately; see the README for longer runs and what they show.